# Chapter 15 - Spin Tracking with Ramping

For a perfectly flat ring, where the closed orbit sees only vertical magnetic fields, the closed-orbit spin tune is

$$
\nu_0 = G\gamma, \qquad G = \frac{g - 2}{2},
$$

with $\gamma$ the reference relativistic gamma. In this example we ramp the proton ring energy through the intrinsic spin-orbit resonance

$$
\nu_0 = 60 - Q_y \approx 51.81.
$$

This notebook translates the Chapter 15 Bmad workflow into SciBmad. The original example uses `spin_lat.bmad`, adds a `ramper` that changes the reference energy, tracks one vertically polarized proton for 5000 turns, and compares the final vertical polarization with the Froissart-Stora prediction.

In [1]:
using SciBmad
using CairoMakie
using LinearAlgebra
using Statistics
using Printf

## 15.1 Introduction

Classic Bmad performs this example with two files:

- `spin_lat.bmad`, which defines the AGS-like proton lattice and the energy ramp;
- `long_term_tracking.init`, which turns on ramping, spin tracking, and turn-by-turn output.

In SciBmad, the same pieces become ordinary Julia objects. A time-dependent reference energy is represented with `Time()`, and spin tracking is enabled by passing an initial spin vector or quaternion to `track`.

## Load the Spin Lattice

The lattice used here is SciBmad's translated AGS-like proton lattice, distributed with the SciBmad package at `joinpath(pkgdir(SciBmad), "lattices", "ags.jl")`. It corresponds to bmad's `15_SpinTracking/spin_lat.bmad`: the quadrupoles, bends, proton reference species, and `CSNK`/`WSNK` marker locations are already encoded as Julia/SciBmad objects. Loading this file creates the `ring` object used below.

In [2]:
# This file mirrors the tutorial's 15_SpinTracking/spin_lat.bmad lattice.
spin_lattice_path = joinpath(pkgdir(SciBmad), "lattices", "ags.jl")
include(spin_lattice_path)

@printf("Loaded lattice: %s\n", spin_lattice_path)
@printf("Ring elements: %d\n", length(ring.line))
@printf("Reference species: %s\n", ring.species_ref.name)
@printf("Initial fixed E_ref from lattice file: %.8e eV\n", ring.E_ref)

Ring elements: 543
Reference species: proton
Initial fixed E_ref from lattice file: 2.40490243e+10 eV


## Define the energy ramp

The Bmad example ramps $G\gamma$ from `51.71` to `51.91` in 5000 turns. Since $G\gamma = G E/(mc^2)$ for the reference particle, the reference total energy is

$$
E = \frac{G\gamma}{G} m.
$$

The tutorial uses a revolution time of `2.67024` microseconds. We use the same value so the ramp rate matches the original `ramper` element.

In [3]:
const CH15_SPECIES = Species("proton")
const CH15_G = gyromagnetic_anomaly(CH15_SPECIES)
const CH15_MASS = massof(CH15_SPECIES)

const G_GAMMA_START = 51.71
const G_GAMMA_STOP = 51.91
const N_TURNS = 5000
const TURN_TIME = 2.67024e-6

energy_from_Ggamma(Ggamma) = Ggamma / CH15_G * CH15_MASS
Ggamma_from_energy(E) = CH15_G * E / CH15_MASS

E_START = energy_from_Ggamma(G_GAMMA_START)
E_STOP = energy_from_Ggamma(G_GAMMA_STOP)
dE_dt = (E_STOP - E_START) / (N_TURNS * TURN_TIME)

@printf("G = %.12f\n", CH15_G)
@printf("E_start = %.8e eV\n", E_START)
@printf("E_stop  = %.8e eV\n", E_STOP)
@printf("dE/dt   = %.8e eV/s\n", dE_dt)

G = 1.792847344650
E_start = 2.70620083e+10 eV
E_stop  = 2.71666767e+10 eV
dE/dt   = 7.83962302e+09 eV/s


In [4]:
# SciBmad version of the Bmad ramper.
# During tracking, BeamTracking evaluates Time() from the bunch reference time.
ring.E_ref = E_START + dE_dt * Time()

# At t = 0, the ring starts at G? = 51.71.
@printf("G?(t=0) = %.5f\n", Ggamma_from_energy(ring.E_ref(0.0)))
@printf("G?(t=end) ? %.5f\n", Ggamma_from_energy(ring.E_ref(N_TURNS * TURN_TIME)))

G?(t=0) = 51.71000
G?(t=end) ? 51.91000


## Track one vertically polarized proton

The six orbital coordinates are ordered as

$$
(x, p_x, y, p_y, z, p_z).
$$

The initial spin vector is vertical, $(S_x, S_y, S_z) = (0, 1, 0)$. Passing this spin vector as the third positional argument to `track` tells SciBmad to return nine saved columns: the six orbital coordinates followed by $S_x$, $S_y$, and $S_z$.

In [ ]:
v0 = [0.0, 0.0, 1e-5, 0.0, 2.66, 1.15e-5]
s0 = [0.0, 1.0, 0.0]

history = track(
    ring,
    v0,
    s0;
    n_turns=N_TURNS,
    save_every_n_turns=1,
)

@printf("history size: %s\n", string(size(history)))
@printf("columns: x px y py z pz Sx Sy Sz\n")

In [ ]:
turns = collect(0:N_TURNS)
Sx = vec(history[1, :, 7])
Sy = vec(history[1, :, 8])
Sz = vec(history[1, :, 9])
Ggamma_turn = G_GAMMA_START .+ (G_GAMMA_STOP - G_GAMMA_START) .* turns ./ N_TURNS

@printf("Initial Sy = %.8f\n", first(Sy))
@printf("Final Sy   = %.8f\n", last(Sy))
@printf("Minimum Sy = %.8f\n", minimum(Sy))

In [ ]:
fig = Figure(size=(760, 440))
ax = Axis(fig[1, 1], xlabel="turn", ylabel="S_y")
lines!(ax, turns, Sy, linewidth=2)
hlines!(ax, [last(Sy)], linestyle=:dash, color=:gray40)
fig

In [ ]:
fig = Figure(size=(760, 440))
ax = Axis(fig[1, 1], xlabel="G?", ylabel="S_y")
lines!(ax, Ggamma_turn, Sy, linewidth=2)
vlines!(ax, [51.81], linestyle=:dash, color=:firebrick)
fig

## Froissart-Stora comparison

For a linear crossing of an isolated resonance, the final vertical polarization is approximated by

$$
S_y(\infty) = 2\exp\left(-\frac{\pi \epsilon^2}{2\tilde\alpha}\right) - 1,
$$

where $\epsilon$ is the resonance strength and

$$
\tilde\alpha = \frac{1}{2\pi}\frac{d(G\gamma)}{d\theta}
              = \frac{1}{2\pi}\left[\text{change in }G\gamma\text{ per turn}\right].
$$

The original tutorial obtains $\epsilon$ from Tao with `show spin -ele 0`. In this SciBmad notebook we compute the ramp parameter directly, and we also infer the resonance strength implied by the tracking result.

In [ ]:
alpha_tilde = (G_GAMMA_STOP - G_GAMMA_START) / N_TURNS / (2pi)

function froissart_stora(epsilon, alpha_tilde)
    return 2exp(-pi * epsilon^2 / (2alpha_tilde)) - 1
end

function epsilon_from_final_polarization(Sy_final, alpha_tilde)
    value = (Sy_final + 1) / 2
    if !(0 < value <= 1)
        return NaN
    end
    return sqrt(-2alpha_tilde / pi * log(value))
end

epsilon_tracking = epsilon_from_final_polarization(last(Sy), alpha_tilde)
Sy_reconstructed = froissart_stora(epsilon_tracking, alpha_tilde)

@printf("alpha_tilde = %.8e\n", alpha_tilde)
@printf("epsilon inferred from tracking = %.8e\n", epsilon_tracking)
@printf("Froissart-Stora with inferred epsilon = %.8f\n", Sy_reconstructed)
@printf("tracking final Sy = %.8f\n", last(Sy))

To compare against Tao rather than the tracking-inferred value, set the energy to the resonance point in Tao and run:

```tao
set ele * e_tot = 51.81 / anomalous_moment_of(proton) * mass_of(proton)
show spin -ele 0
```

For this vertical resonance, choose `Xi_sum` or `Xi_diff` according to which fractional expression is closer to zero, then multiply the normalized Tao value by $\sqrt{J_y}$ to get the physical resonance strength $\epsilon$. Substitute that value into `froissart_stora(epsilon, alpha_tilde)` above.

## Check the spin norm

A useful numerical check is that the spin vector stays close to unit length throughout tracking.

In [ ]:
spin_norm = sqrt.(Sx.^2 .+ Sy.^2 .+ Sz.^2)

fig = Figure(size=(760, 360))
ax = Axis(fig[1, 1], xlabel="turn", ylabel="|S| - 1")
lines!(ax, turns, spin_norm .- 1, linewidth=2)
fig

In [ ]:
@printf("maximum | |S| - 1 | = %.3e\n", maximum(abs.(spin_norm .- 1)))